# 🤖 Automated Hyperparameter Tuning & Model Selection
### Using Databricks AutoML — Classification Project
**Dataset:** Wine Quality (built-in via sklearn)
**Task:** Multiclass Classification (wine quality score → Low / Medium / High)
**Tools:** Databricks AutoML, MLflow, Spark, pandas
---
### 📋 Project Notebooks
| # | Notebook Section | Description |
|---|---|---|
| 1 | Setup & Data Prep | Install libs, load data, create Delta table |
| 2 | AutoML Run | Launch AutoML classify() with MLflow tracking |
| 3 | Results Analysis | Compare all trials, inspect best model |
| 4 | Model Registration | Register best model to MLflow Model Registry |
| 5 | Inference | Load registered model & run batch predictions |

## 📦 Section 1 — Setup & Data Preparation


In [0]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

print("✅ Libraries loaded successfully")
print(f"MLflow version: {mlflow.__version__}")

### 1.1 Load & Explore the Wine Dataset

In [0]:
# Load the built-in wine dataset from sklearn
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df["quality_label"] = wine.target  # 0=class_0, 1=class_1, 2=class_2

# Map numeric labels to descriptive names for clarity
label_map = {0: "low", 1: "medium", 2: "high"}
df["quality"] = df["quality_label"].map(label_map)
df = df.drop(columns=["quality_label"])

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:\n{df['quality'].value_counts()}")
print(f"\nFeature summary:\n{df.describe().round(2)}")

display(df.head(10))

### 1.2 Split Data into Train / Test Sets

In [0]:
# Train/test split — 80% train, 20% test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["quality"])

print(f"Training samples : {len(train_df)}")
print(f"Test samples     : {len(test_df)}")

### 1.3 Save Data as Delta Tables
AutoML works best with Delta tables. We register the training set so AutoML can reference it.

In [0]:
# Convert to Spark DataFrames and save as Delta tables
spark_train = spark.createDataFrame(train_df)
spark_test  = spark.createDataFrame(test_df)

# Write to Delta (overwrite mode for idempotency)
spark_train.write.format("delta").mode("overwrite").saveAsTable("wine_quality_train")
spark_test.write.format("delta").mode("overwrite").saveAsTable("wine_quality_test")

print("✅ Delta tables created:")
print("   • wine_quality_train")
print("   • wine_quality_test")

---
## 🚀 Section 2 — Launch Databricks AutoML

AutoML will:
- Automatically preprocess features (imputation, encoding, scaling)
- Try multiple algorithms: LightGBM, XGBoost, Random Forest, Logistic Regression, Decision Tree
- Tune hyperparameters using Hyperopt under the hood
- Log every trial to MLflow with full metrics and artifacts
- Return a summary object with the best model

In [0]:
from databricks import automl

# Launch AutoML classification run
# timeout_minutes controls how long AutoML searches — increase for better results
# primary_metric is what AutoML optimizes across all trials
summary = automl.classify(
    dataset         = spark.table("wine_quality_train"),   # Delta table reference
    target_col      = "quality",                           # Column to predict
    primary_metric  = "f1",                                # Optimize for F1 score
    timeout_minutes = 15,                                  # ⏱ Adjust to Free Edition quota
    experiment_name = "/automl/wine_quality_classification",
    exclude_cols    = [],                                  # Exclude no columns
)

print("\n✅ AutoML run complete!")
print(f"Best trial run ID : {summary.best_trial.mlflow_run_id}")
print(f"Best model        : {summary.best_trial.model_description}")
print(f"Best F1 score     : {summary.best_trial.metrics['val_f1_score']:.4f}")


---
## 📊 Section 3 — Explore & Compare AutoML Results

### 3.1 View All Trial Results

In [0]:
# Extract all trial metrics into a comparison DataFrame
trials_data = []
for trial in summary.trials:
    row = {
        "run_id"           : trial.mlflow_run_id,
        "model_description": trial.model_description,
        "val_f1_score"     : trial.metrics.get("val_f1_score"),
        "val_accuracy"     : trial.metrics.get("val_accuracy"),
        "val_precision"    : trial.metrics.get("val_precision_score"),
        "val_recall"       : trial.metrics.get("val_recall_score"),
        "training_duration": trial.metrics.get("training_duration"),
    }
    trials_data.append(row)

trials_df = pd.DataFrame(trials_data).sort_values("val_f1_score", ascending=False)
trials_df["rank"] = range(1, len(trials_df) + 1)

print(f"Total trials completed: {len(trials_df)}")
display(trials_df)

### 3.2 Inspect the Best Trial

In [0]:
best_trial = summary.best_trial

print("=" * 55)
print("🏆  BEST MODEL SUMMARY")
print("=" * 55)
print(f"  Algorithm  : {best_trial.model_description}")
print(f"  Run ID     : {best_trial.mlflow_run_id}")
print(f"  F1 Score   : {best_trial.metrics.get('val_f1_score', 'N/A'):.4f}")
print(f"  Accuracy   : {best_trial.metrics.get('val_accuracy', 'N/A'):.4f}")
print(f"  Precision  : {best_trial.metrics.get('val_precision_score', 'N/A'):.4f}")
print(f"  Recall     : {best_trial.metrics.get('val_recall_score', 'N/A'):.4f}")
print("=" * 55)
print(f"\n📓 Open best trial notebook:")
print(f"   {best_trial.notebook_url}")
print(f"\n🔬 Open MLflow experiment:")
print(f"   {summary.experiment.artifact_location}")


### 3.3 Load Best Model & Evaluate on Held-Out Test Set

In [0]:
import mlflow.pyfunc
from sklearn.metrics import classification_report, confusion_matrix

# Load the best model from MLflow
best_model_uri = f"runs:/{best_trial.mlflow_run_id}/model"
best_model = mlflow.pyfunc.load_model(best_model_uri)

print(f"✅ Loaded model from: {best_model_uri}")

# Prepare test features and labels
X_test = test_df.drop(columns=["quality"])
y_test = test_df["quality"]

# Generate predictions
y_pred = best_model.predict(X_test)

# Evaluation report
print("\n📊 Classification Report on Test Set:")
print("=" * 55)
print(classification_report(y_test, y_pred))

print("\n📊 Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

### 3.4 (Optional) Feature Importance via SHAP

AutoML-generated notebooks include SHAP support. Enable below for global explainability.

In [0]:
# ⚠️ SHAP is memory-intensive — enable only on adequate compute
# To enable, open the best trial notebook and set: shap_enabled = True
# Or run the snippet below with the sklearn model loaded directly:

# from mlflow.sklearn import load_model as load_sklearn_model
# import shap
#
# sk_model = load_sklearn_model(best_model_uri)
# explainer = shap.TreeExplainer(sk_model)
# shap_values = explainer.shap_values(X_test)
# shap.summary_plot(shap_values, X_test, plot_type="bar")

---
## 🗂️ Section 4 — Register Best Model to MLflow Model Registry

### 4.1 Register the Model


In [0]:
MODEL_NAME = "wine_quality_classifier"

# Register best model from the AutoML run
registered_model = mlflow.register_model(
    model_uri = f"runs:/{best_trial.mlflow_run_id}/model",
    name      = MODEL_NAME,
)

print(f"✅ Model registered:")
print(f"   Name    : {registered_model.name}")
print(f"   Version : {registered_model.version}")

### 4.2 Transition Model to "Staging" Stage


In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

client.transition_model_version_stage(
    name    = MODEL_NAME,
    version = registered_model.version,
    stage   = "Staging",
    archive_existing_versions = True,   # Archive older versions automatically
)

# Add a descriptive tag and annotation
client.set_model_version_tag(
    name    = MODEL_NAME,
    version = registered_model.version,
    key     = "automl_best_trial",
    value   = best_trial.mlflow_run_id,
)

client.update_model_version(
    name        = MODEL_NAME,
    version     = registered_model.version,
    description = (
        f"AutoML best trial. Algorithm: {best_trial.model_description}. "
        f"Val F1: {best_trial.metrics.get('val_f1_score', 'N/A'):.4f}"
    ),
)

print(f"✅ Model version {registered_model.version} transitioned to 'Staging'")


### 4.3 List All Registered Versions

In [0]:
versions = client.search_model_versions(f"name='{MODEL_NAME}'")

version_info = [{
    "version"     : v.version,
    "stage"       : v.current_stage,
    "run_id"      : v.run_id,
    "description" : v.description,
    "created"     : pd.to_datetime(v.creation_timestamp, unit="ms"),
} for v in versions]

display(pd.DataFrame(version_info))

---
## 🔮 Section 5 — Batch Inference with Registered Model

### 5.1 Load Model from Registry & Predict

In [0]:
# Load the Staging model from the registry
staging_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/Staging")

# Run inference on the test set
test_features = test_df.drop(columns=["quality"])
predictions   = staging_model.predict(test_features)

# Combine predictions with original test data
results_df = test_df.copy()
results_df["predicted_quality"] = predictions
results_df["correct"] = results_df["quality"] == results_df["predicted_quality"]

print(f"✅ Batch inference complete on {len(results_df)} samples")
print(f"   Overall Accuracy: {results_df['correct'].mean():.4f}")

display(results_df[["quality", "predicted_quality", "correct"]].head(20))

### 5.2 Save Predictions to Delta Table

In [0]:
# Persist predictions as a Delta table for downstream use
spark_results = spark.createDataFrame(results_df)
spark_results.write.format("delta").mode("overwrite").saveAsTable("wine_quality_predictions")

print("✅ Predictions saved to Delta table: wine_quality_predictions")

---
## ✅ Project Summary

| Step | What was done |
|---|---|
| 1. Data Prep | Loaded Wine dataset, created Delta tables |
| 2. AutoML Run | Launched `automl.classify()`, tried multiple algorithms + hyperparameter configs |
| 3. Results | Compared all trials via MLflow, identified best model |
| 4. Registration | Registered best model to MLflow Model Registry, promoted to Staging |
| 5. Inference | Loaded registered model, ran batch predictions, saved to Delta |

### 🔗 Next Steps
- **Promote to Production**: Use `client.transition_model_version_stage(..., stage="Production")` when validated
- **Enable Model Serving**: Go to Serving tab in Databricks UI → create endpoint from registry
- **SHAP Explainability**: Enable `shap_enabled = True` in the best trial notebook
- **Custom Search**: Clone the best trial notebook and manually tune further
- **CI/CD**: Use Databricks Workflows to schedule retraining pipelines